# Optimization Model: Production Planning Business Problem

This notebook solves a production planning problem using linear programming with PuLP. The goal is to maximize profit while satisfying resource and demand constraints.


## 1. Import Required Libraries

We import PuLP for optimization and pandas/matplotlib for results handling and visualization.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pulp import LpProblem, LpStatus, LpVariable, LpMaximize, lpSum, value

plt.style.use('seaborn-v0_8')


## 2. Define the Business Problem

A furniture manufacturer produces tables and chairs.
The company wants to decide how many tables and chairs to produce each week to maximize profit.

Decision variables:
- `tables`: number of tables produced per week
- `chairs`: number of chairs produced per week

Objective function:
- Maximize total weekly profit = 70 * tables + 40 * chairs

Constraints:
1. Labor available: 80 hours per week
2. Wood available: 120 board-feet per week
3. Demand limits: at most 15 tables and 35 chairs per week


## 3. Problem Data

We organize the production parameters as a data table. This makes the model easier to update and shows the business inputs clearly.

- Profit per unit
- Labor hours required per unit
- Wood board-feet required per unit
- Maximum weekly demand

In [ ]:
products = ['Tables', 'Chairs']
profit = {'Tables': 70, 'Chairs': 40}
labor = {'Tables': 5, 'Chairs': 2}
wood = {'Tables': 8, 'Chairs': 4}
max_demand = {'Tables': 15, 'Chairs': 35}

data = pd.DataFrame({
    'Profit': [profit[p] for p in products],
    'Labor (hours)': [labor[p] for p in products],
    'Wood (board-feet)': [wood[p] for p in products],
    'Max Demand': [max_demand[p] for p in products]
}, index=products)

data

## 4. Formulate the Optimization Model

We translate the business problem into a linear programming model using PuLP. The model uses decision variables, objective coefficients, and resource constraints defined from the data.


In [ ]:
# Define the problem
problem = LpProblem('Furniture_Production', LpMaximize)

# Decision variables
x = {p: LpVariable(p.lower(), lowBound=0, cat='Continuous') for p in products}

# Objective function
problem += lpSum(profit[p] * x[p] for p in products), 'Total_Profit'

# Resource constraints
problem += lpSum(labor[p] * x[p] for p in products) <= 80, 'Labor_Hours'
problem += lpSum(wood[p] * x[p] for p in products) <= 120, 'Wood_Consumption'

# Demand constraints
for p in products:
    problem += x[p] <= max_demand[p], f'Max_{p}'

## 5. Implement the Model in Python with PuLP

The model is defined using PuLP objects. This section highlights the objective and constraints.


## 6. Solve the Optimization Problem

We use PuLP's default solver to find the optimal production plan.


In [ ]:
problem.solve()
status = LpStatus[problem.status]
status


## 7. Extract and Display Results

We retrieve the optimal decision variable values and the objective value.


In [ ]:
optimal_values = {p: value(x[p]) for p in products}
optimal_profit = value(problem.objective)

results = pd.DataFrame({
    'Product': products,
    'Optimal Quantity': [optimal_values[p] for p in products],
    'Profit per Unit': [profit[p] for p in products]
})

results


In [ ]:
print(f'Optimal profit: ${optimal_profit:,.2f}')
print(f'Status: {status}')


## 8. Analyze and Visualize Insights

We interpret the optimal plan and visualize the production mix.


In [ ]:
display(results)

# Analyze constraint usage and slack
constraint_data = []
for name, constraint in problem.constraints.items():
    lhs_value = sum(var.value() * coef for var, coef in constraint.items())
    rhs_value = constraint.constant
    slack = rhs_value - lhs_value
    constraint_data.append({
        'Constraint': name,
        'Left-hand side': lhs_value,
        'Right-hand side': rhs_value,
        'Slack': slack
    })

constraint_df = pd.DataFrame(constraint_data)
constraint_df

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(results['Product'], results['Optimal Quantity'], color=['#1f77b4', '#ff7f0e'])
ax.set_title('Optimal Weekly Production Plan')
ax.set_ylabel('Quantity')
for index, value_qty in enumerate(results['Optimal Quantity']):
    ax.text(index, value_qty + 0.5, int(value_qty), ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 9. Scenario & Sensitivity Analysis

We analyze how changes in labor capacity affect the optimal product mix and profit. This helps the business understand how resource changes influence decisions.

In [ ]:
scenario_labor = [60, 80, 100]
scenario_rows = []
for labor_capacity in scenario_labor:
    scenario = LpProblem(f'Furniture_Production_Labor_{labor_capacity}', LpMaximize)
    y = {p: LpVariable(f"{p.lower()}_{labor_capacity}", lowBound=0, cat='Continuous') for p in products}
    scenario += lpSum(profit[p] * y[p] for p in products)
    scenario += lpSum(labor[p] * y[p] for p in products) <= labor_capacity
    scenario += lpSum(wood[p] * y[p] for p in products) <= 120
    for p in products:
        scenario += y[p] <= max_demand[p]
    scenario.solve()
    scenario_rows.append({
        'Labor capacity': labor_capacity,
        'Tables': value(y['Tables']),
        'Chairs': value(y['Chairs']),
        'Profit': value(scenario.objective)
    })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scenario_df['Labor capacity'], scenario_df['Profit'], marker='o')
ax.set_title('Profit by Labor Capacity')
ax.set_xlabel('Labor capacity (hours)')
ax.set_ylabel('Profit')
ax.grid(True)
plt.tight_layout()
plt.show()

### Key Insights

- The model identifies the most profitable mix given labor, wood, and demand limits.
- The optimal solution uses available resources efficiently while respecting demand caps.
- The constraint analysis shows which resource limits are binding and where slack remains.
- The visualization makes it easy to compare product quantities and support production decisions.
